In [2]:
%pip install mediapipe opencv-python

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import cv2
import mediapipe as mp
import numpy as np
mp_drawing = mp.solutions.drawing_utils
mp_pose = mp.solutions.pose

In [5]:
import mediapipe as mp

# Initialize drawing utilities
mp_drawing = mp.solutions.drawing_utils
drawing_spec = mp_drawing.DrawingSpec(color=(0, 255, 0), thickness=2, circle_radius=2)


In [7]:
for lndmrk in mp_pose.PoseLandmark:
    print(lndmrk)

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32


In [8]:
def calculate_angle(a,b,c):
    a = np.array(a) # First
    b = np.array(b) # Mid
    c = np.array(c) # End
    
    radians = np.arctan2(c[1]-b[1], c[0]-b[0]) - np.arctan2(a[1]-b[1], a[0]-b[0])
    angle = np.abs(radians*180.0/np.pi)
    
    if angle >180.0:
        angle = 360-angle
        
    return angle 

In [12]:

# Initialize Mediapipe Pose
mp_drawing = mp.solutions.drawing_utils
mp_pose = mp.solutions.pose

In [11]:
cap = cv2.VideoCapture(0)

# Initialize counters and stages for curl detection and fatigue monitoring
counter = 0 
stage = None
fatigue_flag = False
rep_threshold = 10  # A threshold for counting reps to monitor fatigue

# Function to calculate the angle between three points
def calculate_angle(a, b, c):
    a = np.array(a)  # First point
    b = np.array(b)  # Middle point
    c = np.array(c)  # End point
    
    radians = np.arctan2(c[1]-b[1], c[0]-b[0]) - np.arctan2(a[1]-b[1], a[0]-b[0])
    angle = np.abs(radians*180.0/np.pi)
    
    if angle > 180.0:
        angle = 360-angle
        
    return angle


In [16]:
import cv2
import mediapipe as mp
import numpy as np

# Initialize Mediapipe Pose
mp_drawing = mp.solutions.drawing_utils
mp_pose = mp.solutions.pose

# Function to calculate angle between three points
def calculate_angle(a, b, c):
    a = np.array(a)  # First point
    b = np.array(b)  # Midpoint
    c = np.array(c)  # Last point
    
    radians = np.arctan2(c[1] - b[1], c[0] - b[0]) - np.arctan2(a[1] - b[1], a[0] - b[0])
    angle = np.abs(radians * 180.0 / np.pi)
    
    if angle > 180.0:
        angle = 360 - angle
        
    return angle

# Video capture setup
cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

# Variables for repetition counting and symmetry
rep_count = 0
is_down = False
threshold = 0.05  # Threshold for horizontal symmetry
symmetry_line = None

with mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5) as pose:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        
        # Recolor the image to RGB for Mediapipe processing
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image.flags.writeable = False
        results = pose.process(image)
        
        # Convert image back to BGR for OpenCV
        image.flags.writeable = True
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
        
        if results.pose_landmarks:
            landmarks = results.pose_landmarks.landmark
            
            # Extract keypoints
            keypoints = {name: [landmarks[getattr(mp_pose.PoseLandmark, name.upper()).value].x,
                                landmarks[getattr(mp_pose.PoseLandmark, name.upper()).value].y]
                         for name in ['left_shoulder', 'right_shoulder', 'left_hip', 'right_hip',
                                      'left_elbow', 'right_elbow', 'left_wrist', 'right_wrist',
                                      'left_knee', 'right_knee', 'left_ankle', 'right_ankle']}
            
            # Calculate angles
            angles = {
                'left_elbow_angle': f"{calculate_angle(keypoints['left_shoulder'], keypoints['left_elbow'], keypoints['left_wrist']):.1f} deg",
                'right_elbow_angle': f"{calculate_angle(keypoints['right_shoulder'], keypoints['right_elbow'], keypoints['right_wrist']):.1f} deg",
                'left_knee_angle': f"{calculate_angle(keypoints['left_hip'], keypoints['left_knee'], keypoints['left_ankle']):.1f} deg",
                'right_knee_angle': f"{calculate_angle(keypoints['right_hip'], keypoints['right_knee'], keypoints['right_ankle']):.1f} deg",
                'left_shoulder_angle': f"{calculate_angle(keypoints['left_hip'], keypoints['left_shoulder'], keypoints['left_elbow']):.1f} deg",
                'right_shoulder_angle': f"{calculate_angle(keypoints['right_hip'], keypoints['right_shoulder'], keypoints['right_elbow']):.1f} deg",
                'left_hip_angle': f"{calculate_angle(keypoints['left_shoulder'], keypoints['left_hip'], keypoints['left_knee']):.1f} deg",
                'right_hip_angle': f"{calculate_angle(keypoints['right_shoulder'], keypoints['right_hip'], keypoints['right_knee']):.1f} deg",
            }
            
            # Symmetry and repetition counting
            left_shoulder_y = keypoints['left_shoulder'][1]
            right_shoulder_y = keypoints['right_shoulder'][1]
            left_hip_y = keypoints['left_hip'][1]
            right_hip_y = keypoints['right_hip'][1]

            # Vertical symmetry (Body bending left/right)
            vertical_symmetry = abs(left_shoulder_y - right_shoulder_y)
            bending_direction = "Neutral"
            if vertical_symmetry > threshold:
                bending_direction = "Left Bend" if left_shoulder_y > right_shoulder_y else "Right Bend"

            # Horizontal symmetry and reps
            current_line = (left_hip_y + right_hip_y) / 2
            if symmetry_line is None:
                symmetry_line = current_line
            if current_line < symmetry_line - threshold:
                is_down = True
            elif current_line > symmetry_line + threshold and is_down:
                is_down = False
                rep_count += 1

            # Display info
            cv2.rectangle(image, (0, 0), (450, 400), (0, 0, 0), -1)
            for idx, (joint, angle) in enumerate(angles.items()):
                cv2.putText(image, f"{joint.replace('_', ' ').title()}: {angle}", 
                            (10, 30 + idx * 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
            cv2.putText(image, f"Bending: {bending_direction}", (10, 300), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)
            cv2.putText(image, f"Reps: {rep_count}", (10, 330), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)
        
        # Render pose landmarks
        mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS)
        
        # Show the processed frame
        cv2.imshow('Symmetry and Joint Angle Detection', image)

        # Exit on 'q'
        if cv2.waitKey(10) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()


In [17]:
import cv2
import mediapipe as mp
import numpy as np
     
# Initialize Mediapipe Pose
mp_drawing = mp.solutions.drawing_utils
mp_pose = mp.solutions.pose

# Function to calculate angle between three points
def calculate_angle(a, b, c):
    a = np.array(a)  # First point
    b = np.array(b)  # Midpoint
    c = np.array(c)  # Last point
    
    radians = np.arctan2(c[1] - b[1], c[0] - b[0]) - np.arctan2(a[1] - b[1], a[0] - b[0])
    angle = np.abs(radians * 180.0 / np.pi)
    
    if angle > 180.0:
        angle = 360 - angle
        
    return angle

# Video capture setup
cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

# Variables for repetition counting
rep_count = 0
is_down = False
motion_status = "Up"

with mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5) as pose:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        
        # Recolor the image to RGB for Mediapipe processing
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image.flags.writeable = False
        results = pose.process(image)
        
        # Convert image back to BGR for OpenCV
        image.flags.writeable = True
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
        
        if results.pose_landmarks:
            landmarks = results.pose_landmarks.landmark
            
            # Extract keypoints
            keypoints = {
                'SHOULDER': [(landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].x + 
                              landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].x) / 2,
                             (landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].y + 
                              landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].y) / 2],
                'HIP': [(landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].x + 
                         landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].x) / 2,
                        (landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].y + 
                         landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].y) / 2],
                'LEFT_ELBOW': [landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].x, 
                               landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].y],
                'RIGHT_ELBOW': [landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value].x, 
                                landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value].y],
                'LEFT_WRIST': [landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value].x, 
                               landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value].y],
                'RIGHT_WRIST': [landmarks[mp_pose.PoseLandmark.RIGHT_WRIST.value].x, 
                                landmarks[mp_pose.PoseLandmark.RIGHT_WRIST.value].y],
                'LEFT_KNEE': [landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].x, 
                              landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].y],
                'RIGHT_KNEE': [landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value].x, 
                               landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value].y],
                'LEFT_ANKLE': [landmarks[mp_pose.PoseLandmark.LEFT_ANKLE.value].x, 
                               landmarks[mp_pose.PoseLandmark.LEFT_ANKLE.value].y],
                'RIGHT_ANKLE': [landmarks[mp_pose.PoseLandmark.RIGHT_ANKLE.value].x, 
                                landmarks[mp_pose.PoseLandmark.RIGHT_ANKLE.value].y]
            }
            
            # Vertical Symmetry Calculation and Drawing
            mid_shoulder = keypoints['SHOULDER']
            mid_hip = keypoints['HIP']
              
            cv2.line(image, (int(mid_shoulder[0] * image.shape[1]), int(mid_shoulder[1] * image.shape[0])),
                     (int(mid_hip[0] * image.shape[1]), int(mid_hip[1] * image.shape[0])), (0, 204, 204), 2)
            
            # Calculate body bend angle and status
            left_shoulder = [landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].x, 
                             landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].y]
            right_shoulder = [landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].x, 
                              landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].y]
            
            body_bend_angle = calculate_angle(left_shoulder, mid_hip, right_shoulder)
            bend_status = "Straight" if body_bend_angle < 15 else "Left Bend" if right_shoulder[1] < left_shoulder[1] else "Right Bend"
            
            # Horizontal Symmetry (for up-down motion detection)
            shoulder_y = mid_shoulder[1]
            if shoulder_y > 0.6 and not is_down:
                is_down = True
                motion_status = "Down"
            elif shoulder_y < 0.5 and is_down:
                is_down = False
                motion_status = "Up"
                rep_count += 1
            
            # Calculate joint angles
            angles = { 
                'Shoulder': calculate_angle(left_shoulder, mid_shoulder, right_shoulder),
                'Hip': calculate_angle(mid_shoulder, mid_hip, keypoints['LEFT_KNEE']),
                'Left Elbow': calculate_angle(left_shoulder, keypoints['LEFT_ELBOW'], keypoints['LEFT_WRIST']),
                'Right Elbow': calculate_angle(right_shoulder, keypoints['RIGHT_ELBOW'], keypoints['RIGHT_WRIST']),
                'Left Knee': calculate_angle(mid_hip, keypoints['LEFT_KNEE'], keypoints['LEFT_ANKLE']),
                'Right Knee': calculate_angle(mid_hip, keypoints['RIGHT_KNEE'], keypoints['RIGHT_ANKLE']),
                'Left Wrist': calculate_angle(left_shoulder, keypoints['LEFT_ELBOW'], keypoints['LEFT_WRIST']),
                'Right Wrist': calculate_angle(right_shoulder, keypoints['RIGHT_ELBOW'], keypoints['RIGHT_WRIST'])
            }
            
            # Display details in a single values bar
            cv2.rectangle(image, (0, 0), (300, 450), (0, 0, 0), -1)  # Adjusted width and height
            cv2.putText(image, f"Reps: {rep_count}", (10, 50), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
            cv2.putText(image, f"Bend: {bend_status} ({body_bend_angle:.1f}deg)", (10, 80), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
            cv2.putText(image, f"Motion: {motion_status}", (10, 110), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
            for idx, (joint, angle) in enumerate(angles.items()):
                cv2.putText(image, f"{joint}: {angle:.1f}deg", 
                            (10, 140 + idx * 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)
        
        # Render pose landmarks
        mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS)
        
        # Show the processed frame
        cv2.imshow('Body Analysis', image)
  
        # Exit on 'q'
        if cv2.waitKey(10) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()


In [26]:
%pip install ultralytics


[notice] A new release of pip is available: 23.2.1 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
from ultralytics import YOLO

# Load YOLO model (you'll need to train it on barbell images or use a pre-trained model)
yolo_model = YOLO('yolov8n.pt')  # Use a pre-trained model or your custom-trained one

def detect_barbell(frame):
    results = yolo_model(frame)
    barbell_box = None
    for r in results:
        boxes = r.boxes
        for box in boxes:
            if box.cls == 0:  # Assuming class 0 is 'barbell' - adjust based on your model
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                barbell_box = (x1, y1, x2-x1, y2-y1)
                break
        if barbell_box:
            break
    return barbell_box

In [20]:
import cv2
import mediapipe as mp
import numpy as np

# Initialize Mediapipe Pose
mp_drawing = mp.solutions.drawing_utils
mp_pose = mp.solutions.pose

# Function to calculate angle between three points
def calculate_angle(a, b, c):
    a = np.array(a)  # First point
    b = np.array(b)  # Midpoint
    c = np.array(c)  # Last point
    
    radians = np.arctan2(c[1] - b[1], c[0] - b[0]) - np.arctan2(a[1] - b[1], a[0] - b[0])
    angle = np.abs(radians * 180.0 / np.pi)
    
    if angle > 180.0:
        angle = 360 - angle
        
    return angle

# Function to detect barbell based on color and wrist proximity
def detect_barbell(frame, left_wrist, right_wrist):
    # Convert to HSV color space for better color detection
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    
    # Define color range for barbell (example: silver/gray - adjust as needed)
    lower_color = np.array([0, 0, 100])    # Lower bound for gray/silver
    upper_color = np.array([180, 40, 255]) # Upper bound for gray/silver
    
    # Create mask
    mask = cv2.inRange(hsv, lower_color, upper_color)
    
    # Apply morphological operations to clean up the mask
    kernel = np.ones((5,5), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    
    # Find contours
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    # Filter contours for barbell
    barbell_box = None
    if left_wrist is not None and right_wrist is not None:
        # Convert wrist coordinates to pixel values
        lw_x, lw_y = int(left_wrist[0] * frame.shape[1]), int(left_wrist[1] * frame.shape[0])
        rw_x, rw_y = int(right_wrist[0] * frame.shape[1]), int(right_wrist[1] * frame.shape[0])
        
        for contour in contours:
            x, y, w, h = cv2.boundingRect(contour)
            aspect_ratio = w / float(h)
            area = cv2.contourArea(contour)
            
            # Check if it's horizontal (wider than tall) and near wrists
            if (area > 200 and aspect_ratio > 2 and h < frame.shape[0] * 0.1):  # Horizontal object
                # Check if barbell spans between wrists
                barbell_center_y = y + h/2
                barbell_left_x = x
                barbell_right_x = x + w
                
                # Ensure barbell is horizontally between wrists and vertically close to them
                if (barbell_left_x <= lw_x <= barbell_right_x and 
                    barbell_left_x <= rw_x <= barbell_right_x and
                    abs(barbell_center_y - lw_y) < 50 and  # Vertical proximity to left wrist
                    abs(barbell_center_y - rw_y) < 50):    # Vertical proximity to right wrist
                    barbell_box = (x, y, w, h)
                    break
    
    return barbell_box

# Video capture setup
cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

# Variables for repetition counting and symmetry
rep_count = 0
is_down = False
motion_status = "Up"
barbell_detected = False
threshold = 0.05  # Threshold for detecting significant vertical asymmetry

with mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5) as pose:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        
        # Recolor the image to RGB for Mediapipe processing
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image.flags.writeable = False
        results = pose.process(image)
        
        # Convert image back to BGR for OpenCV
        image.flags.writeable = True
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
        
        if results.pose_landmarks:
            landmarks = results.pose_landmarks.landmark
            
            # Extract keypoints
            keypoints = {
                'SHOULDER': [(landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].x + 
                              landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].x) / 2,
                             (landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].y + 
                              landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].y) / 2],
                'HIP': [(landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].x + 
                         landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].x) / 2,
                        (landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].y + 
                         landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].y) / 2],
                'LEFT_ELBOW': [landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].x, 
                               landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].y],
                'RIGHT_ELBOW': [landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value].x, 
                                landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value].y],
                'LEFT_WRIST': [landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value].x, 
                               landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value].y],
                'RIGHT_WRIST': [landmarks[mp_pose.PoseLandmark.RIGHT_WRIST.value].x, 
                                landmarks[mp_pose.PoseLandmark.RIGHT_WRIST.value].y],
                'LEFT_KNEE': [landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].x, 
                              landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].y],
                'RIGHT_KNEE': [landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value].x, 
                               landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value].y],
                'LEFT_ANKLE': [landmarks[mp_pose.PoseLandmark.LEFT_ANKLE.value].x, 
                               landmarks[mp_pose.PoseLandmark.LEFT_ANKLE.value].y],
                'RIGHT_ANKLE': [landmarks[mp_pose.PoseLandmark.RIGHT_ANKLE.value].x, 
                                landmarks[mp_pose.PoseLandmark.RIGHT_ANKLE.value].y]
            }
            
            # Detect barbell
            barbell_box = detect_barbell(frame, keypoints['LEFT_WRIST'], keypoints['RIGHT_WRIST'])
            if barbell_box:
                x, y, w, h = barbell_box
                cv2.rectangle(image, (x, y), (x + w, y + h), (0, 255, 255), 2)
                cv2.putText(image, "Barbell", (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 2)
                barbell_detected = True
            else:
                barbell_detected = False
            
            # Vertical Symmetry Calculation and Drawing
            mid_shoulder = keypoints['SHOULDER']
            mid_hip = keypoints['HIP']
              
            cv2.line(image, (int(mid_shoulder[0] * image.shape[1]), int(mid_shoulder[1] * image.shape[0])),
                     (int(mid_hip[0] * image.shape[1]), int(mid_hip[1] * image.shape[0])), (0, 204, 204), 2)
            
            # Calculate body bend angle and status
            left_shoulder = [landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].x, 
                             landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].y]
            right_shoulder = [landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].x, 
                              landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].y]
            
            body_bend_angle = calculate_angle(left_shoulder, mid_hip, right_shoulder)
            bend_status = "Straight" if body_bend_angle < 15 else "Left Bend" if right_shoulder[1] < left_shoulder[1] else "Right Bend"
            
            # Vertical symmetry (Body bending left/right)
            left_shoulder_y = left_shoulder[1]
            right_shoulder_y = right_shoulder[1]
            vertical_symmetry = abs(left_shoulder_y - right_shoulder_y)
            bending_direction = "Neutral"
            if vertical_symmetry > threshold:
                bending_direction = "Left Bend" if left_shoulder_y > right_shoulder_y else "Right Bend"
            
            # Horizontal Symmetry (for up-down motion detection) - Enhanced with barbell
            shoulder_y = mid_shoulder[1]
            if barbell_detected and barbell_box:
                barbell_y = (y + h/2) / image.shape[0]  # Normalize barbell y-position
                if barbell_y > 0.6 and not is_down:
                    is_down = True
                    motion_status = "Down"
                elif barbell_y < 0.5 and is_down:
                    is_down = False
                    motion_status = "Up"
                    rep_count += 1
            else:
                if shoulder_y > 0.6 and not is_down:
                    is_down = True
                    motion_status = "Down"
                elif shoulder_y < 0.5 and is_down:
                    is_down = False
                    motion_status = "Up"
                    rep_count += 1
            
            # Calculate joint angles
            angles = { 
                'Shoulder': calculate_angle(left_shoulder, mid_shoulder, right_shoulder),
                'Hip': calculate_angle(mid_shoulder, mid_hip, keypoints['LEFT_KNEE']),
                'Left Elbow': calculate_angle(left_shoulder, keypoints['LEFT_ELBOW'], keypoints['LEFT_WRIST']),
                'Right Elbow': calculate_angle(right_shoulder, keypoints['RIGHT_ELBOW'], keypoints['RIGHT_WRIST']),
                'Left Knee': calculate_angle(mid_hip, keypoints['LEFT_KNEE'], keypoints['LEFT_ANKLE']),
                'Right Knee': calculate_angle(mid_hip, keypoints['RIGHT_KNEE'], keypoints['RIGHT_ANKLE']),
                'Left Wrist': calculate_angle(left_shoulder, keypoints['LEFT_ELBOW'], keypoints['LEFT_WRIST']),
                'Right Wrist': calculate_angle(right_shoulder, keypoints['RIGHT_ELBOW'], keypoints['RIGHT_WRIST'])
            }
            
            # Display details in a single values bar
            cv2.rectangle(image, (0, 0), (300, 510), (0, 0, 0), -1)  # Adjusted height for additional info
            cv2.putText(image, f"Reps: {rep_count}", (10, 50), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
            cv2.putText(image, f"Bend_Status: {bend_status} ({body_bend_angle:.1f}deg)", (10, 80), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
            cv2.putText(image, f"Motion: {motion_status}", (10, 110), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
            cv2.putText(image, f"Barbell: {'Detected' if barbell_detected else 'Not Detected'}", 
                       (10, 140), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
            cv2.putText(image, f"Symmetry: {vertical_symmetry:.3f}", 
                       (10, 170), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
            cv2.putText(image, f"Bending_Direction: {bending_direction}", 
                       (10, 200), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
            for idx, (joint, angle) in enumerate(angles.items()):
                cv2.putText(image, f"{joint}: {angle:.1f}deg", 
                           (10, 230 + idx * 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)
        
        # Render pose landmarks
        mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS)
        
        # Show the processed frame
        cv2.imshow('Body Analysis with Barbell Detection', image)
  
        # Exit on 'q'
        if cv2.waitKey(10) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

In [21]:
import cv2
import mediapipe as mp
import numpy as np
import requests
import json

# Initialize Mediapipe Pose
mp_drawing = mp.solutions.drawing_utils
mp_pose = mp.solutions.pose

# Function to calculate angle between three points
def calculate_angle(a, b, c):
    a = np.array(a)
    b = np.array(b)
    c = np.array(c)
    radians = np.arctan2(c[1] - b[1], c[0] - b[0]) - np.arctan2(a[1] - b[1], a[0] - b[0])
    angle = np.abs(radians * 180.0 / np.pi)
    if angle > 180.0:
        angle = 360 - angle
    return angle

# Function to detect barbell
def detect_barbell(frame, left_wrist, right_wrist):
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    lower_color = np.array([0, 0, 100])
    upper_color = np.array([180, 40, 255])
    mask = cv2.inRange(hsv, lower_color, upper_color)
    kernel = np.ones((5, 5), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    barbell_box = None
    if left_wrist is not None and right_wrist is not None:
        lw_x, lw_y = int(left_wrist[0] * frame.shape[1]), int(left_wrist[1] * frame.shape[0])
        rw_x, rw_y = int(right_wrist[0] * frame.shape[1]), int(right_wrist[1] * frame.shape[0])
        for contour in contours:
            x, y, w, h = cv2.boundingRect(contour)
            aspect_ratio = w / float(h)
            area = cv2.contourArea(contour)
            if area > 200 and aspect_ratio > 2 and h < frame.shape[0] * 0.1:
                barbell_center_y = y + h/2
                barbell_left_x = x
                barbell_right_x = x + w
                if (barbell_left_x <= lw_x <= barbell_right_x and 
                    barbell_left_x <= rw_x <= barbell_right_x and
                    abs(barbell_center_y - lw_y) < 50 and 
                    abs(barbell_center_y - rw_y) < 50):
                    barbell_box = (x, y, w, h)
                    break
    return barbell_box

# Function to evaluate correctness (example ranges for snatch catch position)
def evaluate_correctness(angles):
    correct_ranges = {
        "shoulder_angle": (160, 180),  # Near straight overhead
        "knees_angle": (90, 120),     # Partial squat
        "back_angle": (10, 30),       # Slight forward lean
        "wrist_angle": (150, 180),    # Straight wrists
        "hips_angle": (90, 120)       # Partial squat
    }
    correctness = {
        "shoulder": 1 if correct_ranges["shoulder_angle"][0] <= angles["shoulder_angle"] <= correct_ranges["shoulder_angle"][1] else 0,
        "knees": 1 if correct_ranges["knees_angle"][0] <= angles["knees_angle"] <= correct_ranges["knees_angle"][1] else 0,
        "back": 1 if correct_ranges["back_angle"][0] <= angles["back_angle"] <= correct_ranges["back_angle"][1] else 0
    }
    return correctness

# Video capture setup
cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

# Variables
rep_count = 0
is_down = False
motion_status = "Up"
barbell_detected = False
threshold = 0.05
catch_detected = False

# Lists to store angles
shoulder_angles = []
hip_angles = []
left_elbow_angles = []
right_elbow_angles = []
left_knee_angles = []
right_knee_angles = []
left_wrist_angles = []
right_wrist_angles = []

with mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5) as pose:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image.flags.writeable = False
        results = pose.process(image)
        image.flags.writeable = True
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
        
        if results.pose_landmarks:
            landmarks = results.pose_landmarks.landmark
            
            keypoints = {
                'SHOULDER': [(landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].x + landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].x) / 2,
                             (landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].y + landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].y) / 2],
                'HIP': [(landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].x + landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].x) / 2,
                        (landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].y + landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].y) / 2],
                'LEFT_ELBOW': [landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].x, landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].y],
                'RIGHT_ELBOW': [landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value].x, landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value].y],
                'LEFT_WRIST': [landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value].x, landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value].y],
                'RIGHT_WRIST': [landmarks[mp_pose.PoseLandmark.RIGHT_WRIST.value].x, landmarks[mp_pose.PoseLandmark.RIGHT_WRIST.value].y],
                'LEFT_KNEE': [landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].x, landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].y],
                'RIGHT_KNEE': [landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value].x, landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value].y],
                'LEFT_ANKLE': [landmarks[mp_pose.PoseLandmark.LEFT_ANKLE.value].x, landmarks[mp_pose.PoseLandmark.LEFT_ANKLE.value].y],
                'RIGHT_ANKLE': [landmarks[mp_pose.PoseLandmark.RIGHT_ANKLE.value].x, landmarks[mp_pose.PoseLandmark.RIGHT_ANKLE.value].y]
            }
            
            # Detect barbell
            barbell_box = detect_barbell(frame, keypoints['LEFT_WRIST'], keypoints['RIGHT_WRIST'])
            if barbell_box:
                x, y, w, h = barbell_box
                cv2.rectangle(image, (x, y), (x + w, y + h), (0, 255, 255), 2)
                cv2.putText(image, "Barbell", (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 2)
                barbell_detected = True
            else:
                barbell_detected = False
            
            mid_shoulder = keypoints['SHOULDER']
            mid_hip = keypoints['HIP']
            cv2.line(image, (int(mid_shoulder[0] * image.shape[1]), int(mid_shoulder[1] * image.shape[0])),
                     (int(mid_hip[0] * image.shape[1]), int(mid_hip[1] * image.shape[0])), (0, 204, 204), 2)
            
            left_shoulder = [landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].x, landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].y]
            right_shoulder = [landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].x, landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].y]
            
            body_bend_angle = calculate_angle(left_shoulder, mid_hip, right_shoulder)
            bend_status = "Straight" if body_bend_angle < 15 else "Left Bend" if right_shoulder[1] < left_shoulder[1] else "Right Bend"
            
            left_shoulder_y = left_shoulder[1]
            right_shoulder_y = right_shoulder[1]
            vertical_symmetry = abs(left_shoulder_y - right_shoulder_y)
            bending_direction = "Neutral" if vertical_symmetry <= threshold else "Left Bend" if left_shoulder_y > right_shoulder_y else "Right Bend"
            
            shoulder_y = mid_shoulder[1]
            if barbell_detected and barbell_box:
                barbell_y = (y + h/2) / image.shape[0]
                if barbell_y > 0.6 and not is_down:
                    is_down = True
                    motion_status = "Down"
                elif barbell_y < 0.5 and is_down:
                    is_down = False
                    motion_status = "Up"
                    rep_count += 1
            else:
                if shoulder_y > 0.6 and not is_down:
                    is_down = True
                    motion_status = "Down"
                elif shoulder_y < 0.5 and is_down:
                    is_down = False
                    motion_status = "Up"
                    rep_count += 1
            
            # Calculate joint angles
            shoulder_angle = calculate_angle(left_shoulder, mid_shoulder, right_shoulder)
            hip_angle = calculate_angle(mid_shoulder, mid_hip, keypoints['LEFT_KNEE'])
            left_elbow_angle = calculate_angle(left_shoulder, keypoints['LEFT_ELBOW'], keypoints['LEFT_WRIST'])
            right_elbow_angle = calculate_angle(right_shoulder, keypoints['RIGHT_ELBOW'], keypoints['RIGHT_WRIST'])
            left_knee_angle = calculate_angle(mid_hip, keypoints['LEFT_KNEE'], keypoints['LEFT_ANKLE'])
            right_knee_angle = calculate_angle(mid_hip, keypoints['RIGHT_KNEE'], keypoints['RIGHT_ANKLE'])
            left_wrist_angle = calculate_angle(left_shoulder, keypoints['LEFT_ELBOW'], keypoints['LEFT_WRIST'])
            right_wrist_angle = calculate_angle(right_shoulder, keypoints['RIGHT_ELBOW'], keypoints['RIGHT_WRIST'])
            
            # Store angles
            shoulder_angles.append(shoulder_angle)
            hip_angles.append(hip_angle)
            left_elbow_angles.append(left_elbow_angle)
            right_elbow_angles.append(right_elbow_angle)
            left_knee_angles.append(left_knee_angle)
            right_knee_angles.append(right_knee_angle)
            left_wrist_angles.append(left_wrist_angle)
            right_wrist_angles.append(right_wrist_angle)
            
            # Detect catch position (example for snatch: barbell overhead, squat position)
            if barbell_detected and barbell_y < 0.3 and shoulder_y > 0.6 and not catch_detected:  # Overhead and squatting
                catch_detected = True
                correctness = evaluate_correctness({
                    "shoulder_angle": shoulder_angle,
                    "knees_angle": (left_knee_angle + right_knee_angle) / 2,
                    "back_angle": body_bend_angle,
                    "wrist_angle": (left_wrist_angle + right_wrist_angle) / 2,
                    "hips_angle": hip_angle
                })
                
                # User data (example, replace with actual user input)
                user_data = {
                    "username": "john_doe",
                    "age": 25,
                    "age_start": 15,
                    "yrs_experience": 10,
                    "sex_encoded": 1,
                    "body_weight": 80,
                    "lifted_weight": 120,
                    "pose_data": {
                        "angles": {
                            "shoulder_angle": np.mean(shoulder_angles),
                            "hip_angle": np.mean(hip_angles),
                            "left_elbow_angle": np.mean(left_elbow_angles),
                            "right_elbow_angle": np.mean(right_elbow_angles),
                            "left_knee_angle": np.mean(left_knee_angles),
                            "right_knee_angle": np.mean(right_knee_angles),
                            "left_wrist_angle": np.mean(left_wrist_angles),
                            "right_wrist_angle": np.mean(right_wrist_angles)
                        },
                        "correctness": correctness
                    }
                }
                
                # Send to Flask API
                try:
                    response = requests.post("http://127.0.0.1:5000/submit_user_data", json=user_data)
                    print(f"API Response: {response.json()}")
                except Exception as e:
                    print(f"Error sending data to API: {e}")
            
            # Reset catch detection after standing up
            if shoulder_y < 0.5:
                catch_detected = False
            
            # Display
            cv2.rectangle(image, (0, 0), (300, 510), (0, 0, 0), -1)
            cv2.putText(image, f"Reps: {rep_count}", (10, 50), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
            cv2.putText(image, f"Bend: {bend_status} ({body_bend_angle:.1f}deg)", (10, 80), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
            cv2.putText(image, f"Motion: {motion_status}", (10, 110), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
            cv2.putText(image, f"Barbell: {'Detected' if barbell_detected else 'Not Detected'}", (10, 140), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
            cv2.putText(image, f"Symmetry: {vertical_symmetry:.3f}", (10, 170), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
            cv2.putText(image, f"Direction: {bending_direction}", (10, 200), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
            for idx, (joint, angle) in enumerate({
                "shoulder_angle": shoulder_angle,
                "hip_angle": hip_angle,
                "left_elbow_angle": left_elbow_angle,
                "right_elbow_angle": right_elbow_angle,
                "left_knee_angle": left_knee_angle,
                "right_knee_angle": right_knee_angle,
                "left_wrist_angle": left_wrist_angle,
                "right_wrist_angle": right_wrist_angle
            }.items()):
                cv2.putText(image, f"{joint}: {angle:.1f}deg", (10, 230 + idx * 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)
        
        mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS)
        cv2.imshow('Body Analysis with Barbell Detection', image)
        
        if cv2.waitKey(10) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

In [22]:
import cv2
import mediapipe as mp
import numpy as np
import requests
import json

# Initialize Mediapipe Pose
mp_drawing = mp.solutions.drawing_utils
mp_pose = mp.solutions.pose

# Function to calculate angle between three points
def calculate_angle(a, b, c):
    a = np.array(a)
    b = np.array(b)
    c = np.array(c)
    radians = np.arctan2(c[1] - b[1], c[0] - b[0]) - np.arctan2(a[1] - b[1], a[0] - b[0])
    angle = np.abs(radians * 180.0 / np.pi)
    if angle > 180.0:
        angle = 360 - angle
    return angle

# Function to detect barbell
def detect_barbell(frame, left_wrist, right_wrist):
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    lower_color = np.array([0, 0, 100])
    upper_color = np.array([180, 40, 255])
    mask = cv2.inRange(hsv, lower_color, upper_color)
    kernel = np.ones((5, 5), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    barbell_box = None
    if left_wrist is not None and right_wrist is not None:
        lw_x, lw_y = int(left_wrist[0] * frame.shape[1]), int(left_wrist[1] * frame.shape[0])
        rw_x, rw_y = int(right_wrist[0] * frame.shape[1]), int(right_wrist[1] * frame.shape[0])
        for contour in contours:
            x, y, w, h = cv2.boundingRect(contour)
            aspect_ratio = w / float(h)
            area = cv2.contourArea(contour)
            if area > 200 and aspect_ratio > 2 and h < frame.shape[0] * 0.1:
                barbell_center_y = y + h/2
                barbell_left_x = x
                barbell_right_x = x + w
                if (barbell_left_x <= lw_x <= barbell_right_x and 
                    barbell_left_x <= rw_x <= barbell_right_x and
                    abs(barbell_center_y - lw_y) < 50 and 
                    abs(barbell_center_y - rw_y) < 50):
                    barbell_box = (x, y, w, h)
                    break
    return barbell_box

# Function to evaluate correctness (example ranges for snatch catch position)
def evaluate_correctness(angles):
    correct_ranges = {
        "shoulder_angle": (160, 180),  # Near straight overhead
        "knees_angle": (90, 120),     # Partial squat
        "back_angle": (10, 30),       # Slight forward lean
        "wrist_angle": (150, 180),    # Straight wrists
        "hips_angle": (90, 120)       # Partial squat
    }
    correctness = {
        "shoulder": 1 if correct_ranges["shoulder_angle"][0] <= angles["shoulder_angle"] <= correct_ranges["shoulder_angle"][1] else 0,
        "knees": 1 if correct_ranges["knees_angle"][0] <= angles["knees_angle"] <= correct_ranges["knees_angle"][1] else 0,
        "back": 1 if correct_ranges["back_angle"][0] <= angles["back_angle"] <= correct_ranges["back_angle"][1] else 0
    }
    return correctness

# Video capture setup
cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

# Variables
rep_count = 0
is_down = False
motion_status = "Up"
barbell_detected = False
threshold = 0.05
catch_detected = False

# Base user data (replace with dynamic input if needed)
base_user_data = {
    "username": "john_doe",
    "age": 25,
    "age_start": 15,
    "yrs_experience": 10,
    "sex_encoded": 1,
    "body_weight": 80,
    "lifted_weight": 120,
    "pose_data": {
        "angles": {
            "shoulder_angle": 0,
            "knees_angle": 0,
            "back_angle": 0,
            "wrist_angle": 0,
            "hips_angle": 0
        },
        "correctness": {
            "shoulder": 0,
            "knees": 0,
            "back": 0
        }
    }
}

with mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5) as pose:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image.flags.writeable = False
        results = pose.process(image)
        image.flags.writeable = True
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
        
        if results.pose_landmarks:
            landmarks = results.pose_landmarks.landmark
            
            keypoints = {
                'SHOULDER': [(landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].x + landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].x) / 2,
                             (landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].y + landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].y) / 2],
                'HIP': [(landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].x + landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].x) / 2,
                        (landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].y + landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].y) / 2],
                'LEFT_ELBOW': [landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].x, landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].y],
                'RIGHT_ELBOW': [landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value].x, landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value].y],
                'LEFT_WRIST': [landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value].x, landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value].y],
                'RIGHT_WRIST': [landmarks[mp_pose.PoseLandmark.RIGHT_WRIST.value].x, landmarks[mp_pose.PoseLandmark.RIGHT_WRIST.value].y],
                'LEFT_KNEE': [landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].x, landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].y],
                'RIGHT_KNEE': [landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value].x, landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value].y],
                'LEFT_ANKLE': [landmarks[mp_pose.PoseLandmark.LEFT_ANKLE.value].x, landmarks[mp_pose.PoseLandmark.LEFT_ANKLE.value].y],
                'RIGHT_ANKLE': [landmarks[mp_pose.PoseLandmark.RIGHT_ANKLE.value].x, landmarks[mp_pose.PoseLandmark.RIGHT_ANKLE.value].y]
            }
            
            # Detect barbell
            barbell_box = detect_barbell(frame, keypoints['LEFT_WRIST'], keypoints['RIGHT_WRIST'])
            if barbell_box:
                x, y, w, h = barbell_box
                cv2.rectangle(image, (x, y), (x + w, y + h), (0, 255, 255), 2)
                cv2.putText(image, "Barbell", (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 2)
                barbell_detected = True
            else:
                barbell_detected = False
            
            mid_shoulder = keypoints['SHOULDER']
            mid_hip = keypoints['HIP']
            cv2.line(image, (int(mid_shoulder[0] * image.shape[1]), int(mid_shoulder[1] * image.shape[0])),
                     (int(mid_hip[0] * image.shape[1]), int(mid_hip[1] * image.shape[0])), (0, 204, 204), 2)
            
            left_shoulder = [landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].x, landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].y]
            right_shoulder = [landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].x, landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].y]
            
            body_bend_angle = calculate_angle(left_shoulder, mid_hip, right_shoulder)
            bend_status = "Straight" if body_bend_angle < 15 else "Left Bend" if right_shoulder[1] < left_shoulder[1] else "Right Bend"
            
            left_shoulder_y = left_shoulder[1]
            right_shoulder_y = right_shoulder[1]
            vertical_symmetry = abs(left_shoulder_y - right_shoulder_y)
            bending_direction = "Neutral" if vertical_symmetry <= threshold else "Left Bend" if left_shoulder_y > right_shoulder_y else "Right Bend"
            
            shoulder_y = mid_shoulder[1]
            if barbell_detected and barbell_box:
                barbell_y = (y + h/2) / image.shape[0]
                if barbell_y > 0.6 and not is_down:
                    is_down = True
                    motion_status = "Down"
                elif barbell_y < 0.5 and is_down:
                    is_down = False
                    motion_status = "Up"
                    rep_count += 1
            else:
                if shoulder_y > 0.6 and not is_down:
                    is_down = True
                    motion_status = "Down"
                elif shoulder_y < 0.5 and is_down:
                    is_down = False
                    motion_status = "Up"
                    rep_count += 1
            
            # Calculate joint angles
            angles = {
                "shoulder_angle": calculate_angle(left_shoulder, mid_shoulder, right_shoulder),
                "knees_angle": (calculate_angle(mid_hip, keypoints['LEFT_KNEE'], keypoints['LEFT_ANKLE']) + 
                                calculate_angle(mid_hip, keypoints['RIGHT_KNEE'], keypoints['RIGHT_ANKLE'])) / 2,
                "back_angle": body_bend_angle,
                "wrist_angle": (calculate_angle(left_shoulder, keypoints['LEFT_ELBOW'], keypoints['LEFT_WRIST']) + 
                                calculate_angle(right_shoulder, keypoints['RIGHT_ELBOW'], keypoints['RIGHT_WRIST'])) / 2,
                "hips_angle": calculate_angle(mid_shoulder, mid_hip, keypoints['LEFT_KNEE'])
            }
            
            # Detect catch position (e.g., snatch: barbell overhead, squat position)
            if barbell_detected and barbell_y < 0.3 and shoulder_y > 0.6 and not catch_detected:
                catch_detected = True
                correctness = evaluate_correctness(angles)
                
                # Update payload with catch position data
                user_data = base_user_data.copy()  # Create a copy to avoid modifying the base
                user_data["pose_data"]["angles"] = {
                    "shoulder_angle": angles["shoulder_angle"],
                    "knees_angle": angles["knees_angle"],
                    "back_angle": angles["back_angle"],
                    "wrist_angle": angles["wrist_angle"],
                    "hips_angle": angles["hips_angle"]
                }
                user_data["pose_data"]["correctness"] = correctness
                
                # Send to Flask API
                try:
                    response = requests.post("http://127.0.0.1:5000/submit_user_data", json=user_data)
                    print(f"API Response: {response.json()}")
                except Exception as e:
                    print(f"Error sending data to API: {e}")
            
            # Reset catch detection after standing up
            if shoulder_y < 0.5:
                catch_detected = False
            
            # Display
            cv2.rectangle(image, (0, 0), (300, 510), (0, 0, 0), -1)
            cv2.putText(image, f"Reps: {rep_count}", (10, 50), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
            cv2.putText(image, f"Bend: {bend_status} ({body_bend_angle:.1f}deg)", (10, 80), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
            cv2.putText(image, f"Motion: {motion_status}", (10, 110), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
            cv2.putText(image, f"Barbell: {'Detected' if barbell_detected else 'Not Detected'}", (10, 140), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
            cv2.putText(image, f"Symmetry: {vertical_symmetry:.3f}", (10, 170), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
            cv2.putText(image, f"Direction: {bending_direction}", (10, 200), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
            for idx, (joint, angle) in enumerate(angles.items()):
                cv2.putText(image, f"{joint}: {angle:.1f}deg", (10, 230 + idx * 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)
        
        mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS)
        cv2.imshow('Body Analysis with Barbell Detection', image)
        
        if cv2.waitKey(10) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

In [1]:
import cv2
import mediapipe as mp
import numpy as np
import requests
import json
import time

# Initialize Mediapipe Pose
mp_drawing = mp.solutions.drawing_utils
mp_pose = mp.solutions.pose

# Function to calculate angle between three points
def calculate_angle(a, b, c):
    a = np.array(a)
    b = np.array(b)
    c = np.array(c)
    radians = np.arctan2(c[1] - b[1], c[0] - b[0]) - np.arctan2(a[1] - b[1], a[0] - b[0])
    angle = np.abs(radians * 180.0 / np.pi)
    if angle > 180.0:
        angle = 360 - angle
    return angle

# Function to detect barbell
def detect_barbell(frame, left_wrist, right_wrist):
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    lower_color = np.array([0, 0, 100])
    upper_color = np.array([180, 40, 255])
    mask = cv2.inRange(hsv, lower_color, upper_color)
    kernel = np.ones((5, 5), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    barbell_box = None
    if left_wrist is not None and right_wrist is not None:
        lw_x, lw_y = int(left_wrist[0] * frame.shape[1]), int(left_wrist[1] * frame.shape[0])
        rw_x, rw_y = int(right_wrist[0] * frame.shape[1]), int(right_wrist[1] * frame.shape[0])
        for contour in contours:
            x, y, w, h = cv2.boundingRect(contour)
            aspect_ratio = w / float(h)
            area = cv2.contourArea(contour)
            if area > 200 and aspect_ratio > 2 and h < frame.shape[0] * 0.1:
                barbell_center_y = y + h/2
                barbell_left_x = x
                barbell_right_x = x + w
                if (barbell_left_x <= lw_x <= barbell_right_x and 
                    barbell_left_x <= rw_x <= barbell_right_x and
                    abs(barbell_center_y - lw_y) < 50 and 
                    abs(barbell_center_y - rw_y) < 50):
                    barbell_box = (x, y, w, h)
                    break
    return barbell_box

# Function to evaluate correctness
def evaluate_correctness(angles):
    correct_ranges = {
        "shoulder_angle": (160, 180),  # Near straight overhead
        "knees_angle": (90, 120),     # Partial squat
        "back_angle": (10, 30),       # Slight forward lean
        "wrist_angle": (150, 180),    # Straight wrists
        "hips_angle": (90, 120)       # Partial squat
    }
    correctness = {
        "shoulder": 1 if correct_ranges["shoulder_angle"][0] <= angles["shoulder_angle"] <= correct_ranges["shoulder_angle"][1] else 0,
        "knees": 1 if correct_ranges["knees_angle"][0] <= angles["knees_angle"] <= correct_ranges["knees_angle"][1] else 0,
        "back": 1 if correct_ranges["back_angle"][0] <= angles["back_angle"] <= correct_ranges["back_angle"][1] else 0
    }
    return correctness

# Video capture setup
cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

# Variables
rep_count = 0
is_down = False
motion_status = "Up"
barbell_detected = False
threshold = 0.05
catch_count = 0
start_time = None
catch_detected = False

# Base user data
base_user_data = {
    "username": "john_doe",
    "age": 25,
    "age_start": 15,
    "yrs_experience": 10,
    "sex_encoded": 1,
    "body_weight": 80,
    "lifted_weight": 120,
    "pose_data": {
        "angles": {
            "shoulder_angle": 0,
            "knees_angle": 0,
            "back_angle": 0,
            "wrist_angle": 0,
            "hips_angle": 0
        },
        "correctness": {
            "shoulder": 0,
            "knees": 0,
            "back": 0
        }
    }
}

with mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5) as pose:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image.flags.writeable = False
        results = pose.process(image)
        image.flags.writeable = True
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
        
        if results.pose_landmarks:
            landmarks = results.pose_landmarks.landmark
            
            keypoints = {
                'SHOULDER': [(landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].x + landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].x) / 2,
                             (landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].y + landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].y) / 2],
                'HIP': [(landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].x + landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].x) / 2,
                        (landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].y + landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].y) / 2],
                'LEFT_ELBOW': [landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].x, landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].y],
                'RIGHT_ELBOW': [landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value].x, landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value].y],
                'LEFT_WRIST': [landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value].x, landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value].y],
                'RIGHT_WRIST': [landmarks[mp_pose.PoseLandmark.RIGHT_WRIST.value].x, landmarks[mp_pose.PoseLandmark.RIGHT_WRIST.value].y],
                'LEFT_KNEE': [landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].x, landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].y],
                'RIGHT_KNEE': [landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value].x, landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value].y],
                'LEFT_ANKLE': [landmarks[mp_pose.PoseLandmark.LEFT_ANKLE.value].x, landmarks[mp_pose.PoseLandmark.LEFT_ANKLE.value].y],
                'RIGHT_ANKLE': [landmarks[mp_pose.PoseLandmark.RIGHT_ANKLE.value].x, landmarks[mp_pose.PoseLandmark.RIGHT_ANKLE.value].y]
            }
            
            # Detect barbell
            barbell_box = detect_barbell(frame, keypoints['LEFT_WRIST'], keypoints['RIGHT_WRIST'])
            if barbell_box:
                x, y, w, h = barbell_box
                cv2.rectangle(image, (x, y), (x + w, y + h), (255, 0, 0), 2)  # Change color to blue
                cv2.putText(image, "Barbell", (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2)  # Change color to blue
                barbell_detected = True
            else:
                barbell_detected = False
            
            mid_shoulder = keypoints['SHOULDER']
            mid_hip = keypoints['HIP']
            cv2.line(image, (int(mid_shoulder[0] * image.shape[1]), int(mid_shoulder[1] * image.shape[0])),
                     (int(mid_hip[0] * image.shape[1]), int(mid_hip[1] * image.shape[0])), (0, 204, 204), 2)
            
            left_shoulder = [landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].x, landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].y]
            right_shoulder = [landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].x, landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].y]
            
            body_bend_angle = calculate_angle(left_shoulder, mid_hip, right_shoulder)
            bend_status = "Straight" if body_bend_angle < 15 else "Left Bend" if right_shoulder[1] < left_shoulder[1] else "Right Bend"
            
            left_shoulder_y = left_shoulder[1]
            right_shoulder_y = right_shoulder[1]
            vertical_symmetry = abs(left_shoulder_y - right_shoulder_y)
            bending_direction = "Neutral" if vertical_symmetry <= threshold else "Left Bend" if left_shoulder_y > right_shoulder_y else "Right Bend"
            
            shoulder_y = mid_shoulder[1]
            if barbell_detected and barbell_box:
                barbell_y = (y + h/2) / image.shape[0]
                if barbell_y > 0.6 and not is_down:
                    is_down = True
                    motion_status = "Down"
                elif barbell_y < 0.5 and is_down:
                    is_down = False
                    motion_status = "Up"
                    rep_count += 1
            else:
                if shoulder_y > 0.6 and not is_down:
                    is_down = True
                    motion_status = "Down"
                elif shoulder_y < 0.5 and is_down:
                    is_down = False
                    motion_status = "Up"
                    rep_count += 1
            
            # Calculate joint angles
            angles = {
                "shoulder_angle": calculate_angle(left_shoulder, mid_shoulder, right_shoulder),
                "knees_angle": (calculate_angle(mid_hip, keypoints['LEFT_KNEE'], keypoints['LEFT_ANKLE']) + 
                                calculate_angle(mid_hip, keypoints['RIGHT_KNEE'], keypoints['RIGHT_ANKLE'])) / 2,
                "back_angle": body_bend_angle,
                "wrist_angle": (calculate_angle(left_shoulder, keypoints['LEFT_ELBOW'], keypoints['LEFT_WRIST']) + 
                                calculate_angle(right_shoulder, keypoints['RIGHT_ELBOW'], keypoints['RIGHT_WRIST'])) / 2,
                "hips_angle": calculate_angle(mid_shoulder, mid_hip, keypoints['LEFT_KNEE'])
            }
            
            # Detect catch position
            if barbell_detected and barbell_y < 0.3 and shoulder_y > 0.6 and not catch_detected:
                catch_detected = True
                if start_time is None:
                    start_time = time.time()  # Start the timer on first catch
                
                catch_count += 1
                correctness = evaluate_correctness(angles)
                
                # Update payload with catch position data
                user_data = base_user_data.copy()
                user_data["pose_data"]["angles"] = angles
                user_data["pose_data"]["correctness"] = correctness
                
                # For testing: Store immediately
                # For production: Store after second catch within 30-40s
                elapsed_time = time.time() - start_time if start_time else 0
                store_data = False
                
                # Testing mode: Store on every catch
                store_data = True  # Comment this out for production
                
                # Production mode: Store after second catch within 30-40s
                # if catch_count == 2 and 30 <= elapsed_time <= 40:
                #     store_data = True
                
                if store_data:
                    try:
                        response = requests.post("http://127.0.0.1:5000/submit_user_data", json=user_data)
                        print(f"API Response: {response.json()}")
                    except Exception as e:
                        print(f"Error sending data to API: {e}")
            
            # Reset catch detection and timer
            if shoulder_y < 0.5:
                catch_detected = False
                if catch_count >= 2 or (start_time and time.time() - start_time > 40):
                    start_time = None  # Reset timer after second catch or timeout
                    catch_count = 0
            
            # Display
            cv2.rectangle(image, (0, 0), (300, 510), (0, 0, 0), -1)
            cv2.putText(image, f"Reps: {rep_count}", (10, 50), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
            cv2.putText(image, f"Bend: {bend_status} ({body_bend_angle:.1f}deg)", (10, 80), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
            cv2.putText(image, f"Motion: {motion_status}", (10, 110), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
            cv2.putText(image, f"Barbell: {'Detected' if barbell_detected else 'Not Detected'}", (10, 140), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
            cv2.putText(image, f"Symmetry: {vertical_symmetry:.3f}", (10, 170), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
            cv2.putText(image, f"Direction: {bending_direction}", (10, 200), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
            cv2.putText(image, f"Catch Count: {catch_count}", (10, 230), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
            for idx, (joint, angle) in enumerate(angles.items()):
                cv2.putText(image, f"{joint}: {angle:.1f}deg", (10, 260 + idx * 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)
        
        mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS)
        cv2.imshow('Body Analysis with Barbell Detection', image)
        
        if cv2.waitKey(10) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

c:\Users\sdanw\AppData\Local\Programs\Python\Python312\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


In [24]:
import cv2
import mediapipe as mp
import numpy as np
import requests
import json
import time  # Import time for handling session timing

# Initialize Mediapipe Pose
mp_drawing = mp.solutions.drawing_utils
mp_pose = mp.solutions.pose

# Function to calculate angle between three points
def calculate_angle(a, b, c):
    a, b, c = np.array(a), np.array(b), np.array(c)
    radians = np.arctan2(c[1] - b[1], c[0] - b[0]) - np.arctan2(a[1] - b[1], a[0] - b[0])
    angle = np.abs(radians * 180.0 / np.pi)
    return 360 - angle if angle > 180.0 else angle

# Function to evaluate correctness
def evaluate_correctness(angles):
    correct_ranges = {
        "shoulder_angle": (160, 180),
        "knees_angle": (90, 120),
        "back_angle": (10, 30),
        "wrist_angle": (150, 180),
        "hips_angle": (90, 120)
    }
    return {
        key: 1 if correct_ranges[key][0] <= angles[key] <= correct_ranges[key][1] else 0
        for key in correct_ranges
    }

# Start Video Capture
cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

# Start Timer
start_time = time.time()
session_duration = 30  # Set session duration to 30 seconds

# Variables
rep_count = 0
is_down = False
motion_status = "Up"
angles_list = []

# Base user data (Modify as needed)
user_data = {
    "username": "sahan02",
    "age": 45,
    "age_start": 15,
    "yrs_experience": 10,
    "sex_encoded": 0,
    "body_weight": 70,
    "lifted_weight": 80,
    "pose_data": {
        "angles": {},
        "correctness": {}
    }
}

with mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5) as pose:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        
        # Process the frame
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = pose.process(image)
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
        
        if results.pose_landmarks:
            landmarks = results.pose_landmarks.landmark

            keypoints = {
                'SHOULDER': [(landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].x + 
                              landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].x) / 2,
                             (landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].y + 
                              landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].y) / 2],
                'HIP': [(landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].x + 
                         landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].x) / 2,
                        (landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].y + 
                         landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].y) / 2],
                'LEFT_KNEE': [landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].x, landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].y],
                'RIGHT_KNEE': [landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value].x, landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value].y],
                'LEFT_ANKLE': [landmarks[mp_pose.PoseLandmark.LEFT_ANKLE.value].x, landmarks[mp_pose.PoseLandmark.LEFT_ANKLE.value].y],
                'RIGHT_ANKLE': [landmarks[mp_pose.PoseLandmark.RIGHT_ANKLE.value].x, landmarks[mp_pose.PoseLandmark.RIGHT_ANKLE.value].y]
            }

            # Calculate angles
            angles = {
                "shoulder_angle": calculate_angle(keypoints['HIP'], keypoints['SHOULDER'], keypoints['LEFT_KNEE']),
                "knees_angle": (calculate_angle(keypoints['HIP'], keypoints['LEFT_KNEE'], keypoints['LEFT_ANKLE']) +
                                calculate_angle(keypoints['HIP'], keypoints['RIGHT_KNEE'], keypoints['RIGHT_ANKLE'])) / 2,
                "back_angle": calculate_angle(keypoints['SHOULDER'], keypoints['HIP'], keypoints['LEFT_KNEE']),
                "wrist_angle": 170,  # Placeholder since wrist data isn't included
                "hips_angle": calculate_angle(keypoints['SHOULDER'], keypoints['HIP'], keypoints['LEFT_KNEE'])
            }

            # Store angles for later submission
            angles_list.append(angles)

            # Motion tracking
            shoulder_y = keypoints['SHOULDER'][1]
            if shoulder_y > 0.6 and not is_down:
                is_down = True
                motion_status = "Down"
            elif shoulder_y < 0.5 and is_down:
                is_down = False
                motion_status = "Up"
                rep_count += 1

            # Display Data on Screen
            cv2.rectangle(image, (0, 0), (300, 510), (0, 0, 0), -1)
            cv2.putText(image, f"Reps: {rep_count}", (10, 50), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
            cv2.putText(image, f"Motion: {motion_status}", (10, 80), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

            for idx, (joint, angle) in enumerate(angles.items()):
                cv2.putText(image, f"{joint}: {angle:.1f} deg", (10, 110 + idx * 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)

        mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS)
        cv2.imshow('Pose Tracking System', image)
        
        # Stop session after 30 seconds
        if time.time() - start_time > session_duration:
            print("⏳ Session ended. Sending data to database...")
            break

        # Press 'q' to exit manually
        if cv2.waitKey(10) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

# Calculate average angles for final submission
final_angles = {key: sum(d[key] for d in angles_list) / len(angles_list) for key in angles_list[0]}
correctness = evaluate_correctness(final_angles)

# Prepare Data for API Submission
user_data["pose_data"]["angles"] = final_angles
user_data["pose_data"]["correctness"] = correctness

# Send to Flask API
try:
    response = requests.post("http://127.0.0.1:5000/submit_user_data", json=user_data)
    print(f"✅ Data Sent Successfully! API Response: {response.json()}")
except Exception as e:
    print(f"❌ Error sending data to API: {e}")


⏳ Session ended. Sending data to database...
❌ Error sending data to API: HTTPConnectionPool(host='127.0.0.1', port=5000): Max retries exceeded with url: /submit_user_data (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x000002535B3EADE0>: Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it'))


In [27]:
import cv2
import mediapipe as mp
import numpy as np
import requests
import json
import time

# Initialize Mediapipe Pose
mp_drawing = mp.solutions.drawing_utils
mp_pose = mp.solutions.pose

# Function to calculate angle between three points
def calculate_angle(a, b, c):
    a, b, c = np.array(a), np.array(b), np.array(c)
    radians = np.arctan2(c[1] - b[1], c[0] - b[0]) - np.arctan2(a[1] - b[1], a[0] - b[0])
    angle = np.abs(radians * 180.0 / np.pi)
    return 360 - angle if angle > 180.0 else angle

# Function to evaluate correctness based on predefined ranges
def evaluate_correctness(angles):
    correct_ranges = {
        "shoulder_angle": (160, 180),
        "knees_angle": (90, 120),
        "back_angle": (10, 30),
        "wrist_angle": (150, 180),
        "hips_angle": (90, 120)
    }
    return {
        key: 1 if correct_ranges[key][0] <= angles[key] <= correct_ranges[key][1] else 0
        for key in correct_ranges
    }

# Initialize Video Capture
cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

# Start Timer for 30 Seconds Session
start_time = time.time()
session_duration = 30  # Set session duration to 30 seconds

# Variables
rep_count = 0
is_down = False
motion_status = "Up"
angles_list = []

# Base user data (Modify as needed)
user_data = {
    "username": "sahan03",
    "age": 35,
    "age_start": 20,
    "yrs_experience": 15,
    "sex_encoded": 1,
    "body_weight": 130,
    "lifted_weight": 160,
    "pose_data": {
        "angles": {},
        "correctness": {}
    }
}

with mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5) as pose:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        
        # Convert frame to RGB for Mediapipe
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = pose.process(image)
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
        
        if results.pose_landmarks:
            landmarks = results.pose_landmarks.landmark

            keypoints = {
                'SHOULDER': [(landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].x + 
                              landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].x) / 2,
                             (landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].y + 
                              landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].y) / 2],
                'HIP': [(landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].x + 
                         landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].x) / 2,
                        (landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].y + 
                         landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].y) / 2],
                'LEFT_KNEE': [landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].x, landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].y],
                'RIGHT_KNEE': [landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value].x, landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value].y],
                'LEFT_ANKLE': [landmarks[mp_pose.PoseLandmark.LEFT_ANKLE.value].x, landmarks[mp_pose.PoseLandmark.LEFT_ANKLE.value].y],
                'RIGHT_ANKLE': [landmarks[mp_pose.PoseLandmark.RIGHT_ANKLE.value].x, landmarks[mp_pose.PoseLandmark.RIGHT_ANKLE.value].y]
            }

            # Calculate angles
            angles = {
                "shoulder_angle": calculate_angle(keypoints['HIP'], keypoints['SHOULDER'], keypoints['LEFT_KNEE']),
                "knees_angle": (calculate_angle(keypoints['HIP'], keypoints['LEFT_KNEE'], keypoints['LEFT_ANKLE']) +
                                calculate_angle(keypoints['HIP'], keypoints['RIGHT_KNEE'], keypoints['RIGHT_ANKLE'])) / 2,
                "back_angle": calculate_angle(keypoints['SHOULDER'], keypoints['HIP'], keypoints['LEFT_KNEE']),
                "wrist_angle": 170,  # Placeholder for missing wrist tracking
                "hips_angle": calculate_angle(keypoints['SHOULDER'], keypoints['HIP'], keypoints['LEFT_KNEE'])
            }

            # Store angles for later submission
            angles_list.append(angles)

            # Motion Tracking for Reps
            shoulder_y = keypoints['SHOULDER'][1]
            if shoulder_y > 0.6 and not is_down:
                is_down = True
                motion_status = "Down"
            elif shoulder_y < 0.5 and is_down:
                is_down = False
                motion_status = "Up"
                rep_count += 1

            # Display Data on Screen
            cv2.rectangle(image, (0, 0), (300, 510), (0, 0, 0), -1)
            cv2.putText(image, f"Reps: {rep_count}", (10, 50), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
            cv2.putText(image, f"Motion: {motion_status}", (10, 80), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

            for idx, (joint, angle) in enumerate(angles.items()):
                cv2.putText(image, f"{joint}: {angle:.1f} deg", (10, 110 + idx * 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)

        # Draw pose landmarks
        mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS)
        cv2.imshow('Pose Tracking System', image)
        
        # Stop session after 30 seconds
        if time.time() - start_time > session_duration:
            print("⏳ Session ended. Sending data to database...")
            break

        # Press 'q' to exit manually
        if cv2.waitKey(10) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

# Calculate average angles for final submission
final_angles = {key: sum(d[key] for d in angles_list) / len(angles_list) for key in angles_list[0]}
correctness = evaluate_correctness(final_angles)

# Prepare Data for API Submission
user_data["pose_data"]["angles"] = final_angles
user_data["pose_data"]["correctness"] = correctness

# Send Data to Flask API
try:
    response = requests.post("http://127.0.0.1:5000/submit_user_data", json=user_data)
    print(f"✅ Data Sent Successfully! API Response: {response.json()}")
except Exception as e:
    print(f"❌ Error sending data to API: {e}")


❌ Error sending data to API: HTTPConnectionPool(host='127.0.0.1', port=5000): Max retries exceeded with url: /submit_user_data (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x000002535A97C7D0>: Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it'))


In [3]:
import cv2
import mediapipe as mp
import numpy as np
import requests
import json
import time

# Initialize Mediapipe Pose
mp_drawing = mp.solutions.drawing_utils
mp_pose = mp.solutions.pose

# Function to calculate angle between three points
def calculate_angle(a, b, c):
    a, b, c = np.array(a), np.array(b), np.array(c)
    radians = np.arctan2(c[1] - b[1], c[0] - b[0]) - np.arctan2(a[1] - b[1], a[0] - b[0])
    angle = np.abs(radians * 180.0 / np.pi)
    return 360 - angle if angle > 180.0 else angle

# Function to evaluate injury risk
def evaluate_injury_risk(angles):
    risk_levels = {}
    
    risk_ranges = {
        "shoulder_angle": {"low": (160, 180), "moderate": (140, 160, 180, 185), "high": (0, 140, 185, 360)},
        "knees_angle": {"low": (90, 120), "moderate": (70, 90, 120, 140), "high": (0, 70, 140, 360)},
        "back_angle": {"low": (10, 30), "moderate": (5, 10, 30, 40), "high": (0, 5, 40, 360)},
        "wrist_angle": {"low": (150, 180), "moderate": (130, 150, 180, 190), "high": (0, 130, 190, 360)},
        "hips_angle": {"low": (90, 120), "moderate": (70, 90, 120, 140), "high": (0, 70, 140, 360)}
    }
    
    for joint, angle in angles.items():
        if risk_ranges[joint]["low"][0] <= angle <= risk_ranges[joint]["low"][1]:
            risk_levels[joint] = "🟢 Low Risk"
        elif (risk_ranges[joint]["moderate"][0] <= angle <= risk_ranges[joint]["moderate"][1]) or \
             (risk_ranges[joint]["moderate"][2] <= angle <= risk_ranges[joint]["moderate"][3]):
            risk_levels[joint] = "🟠 Moderate Risk"
        else:
            risk_levels[joint] = "🔴 High Risk"

    return risk_levels

# Initialize Video Capture
cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

# Start Timer for 30 Seconds Session
start_time = time.time()
session_duration = 30  

# Variables
rep_count = 0
is_down = False
motion_status = "Up"
angles_list = []

# Base user data (Modify as needed)
user_data = {
    "username": "sahan03",
    "age": 35,
    "age_start": 20,
    "yrs_experience": 15,
    "sex_encoded": 1,
    "body_weight": 130,
    "lifted_weight": 160,
    "pose_data": {
        "angles": {},
        "injury_risk": {}
    }
}

with mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5) as pose:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = pose.process(image)
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
        
        if results.pose_landmarks:
            landmarks = results.pose_landmarks.landmark

            keypoints = {
                'SHOULDER': [(landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].x + 
                              landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].x) / 2,
                             (landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].y + 
                              landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].y) / 2],
                'HIP': [(landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].x + 
                         landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].x) / 2,
                        (landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].y + 
                         landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].y) / 2],
                'LEFT_KNEE': [landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].x, landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].y],
                'RIGHT_KNEE': [landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value].x, landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value].y]
            }

            angles = {
                "shoulder_angle": calculate_angle(keypoints['HIP'], keypoints['SHOULDER'], keypoints['LEFT_KNEE']),
                "knees_angle": calculate_angle(keypoints['HIP'], keypoints['LEFT_KNEE'], keypoints['RIGHT_KNEE']),
                "back_angle": calculate_angle(keypoints['SHOULDER'], keypoints['HIP'], keypoints['LEFT_KNEE']),
                "wrist_angle": 170,  
                "hips_angle": calculate_angle(keypoints['SHOULDER'], keypoints['HIP'], keypoints['LEFT_KNEE'])
            }

            angles_list.append(angles)

            injury_risk = evaluate_injury_risk(angles)

        mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS)
        cv2.imshow('Injury Risk Detection', image)
        
        if time.time() - start_time > session_duration:
            break

        if cv2.waitKey(10) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

# Calculate average angles and injury risk
final_angles = {key: sum(d[key] for d in angles_list) / len(angles_list) for key in angles_list[0]}
user_data["pose_data"]["angles"] = final_angles
user_data["pose_data"]["injury_risk"] = evaluate_injury_risk(final_angles)

# Send Data to Flask API
try:
    response = requests.post("http://127.0.0.1:5000/submit_user_data", json=user_data)
    print(f"✅ Data Sent Successfully! API Response: {response.json()}")
except Exception as e:
    print(f"❌ Error sending data to API: {e}")


❌ Error sending data to API: HTTPConnectionPool(host='127.0.0.1', port=5000): Max retries exceeded with url: /submit_user_data (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x0000019B160DED20>: Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it'))
